# 전체 PDF 청킹 미리보기

이번 노트북은 PDF를 읽고 페이지별 글을 메모리에서 청크로 나눠 봅니다. 청크 파일이나 Supabase에는 저장하지 않습니다. 우선 같은 글자 수 규칙을 적용하고 일부 페이지를 확인합니다. 그림 설명은 사용자가 확인해 준 그림만 넣으며, 모르는 그림은 추측하지 않습니다.

## 수업 기반과 프로젝트 추가 기록 (2026-10-01)

코드의 추가 설명은 주석으로 남겼습니다. 이 표시는 프로젝트 적용 기록이며, 수업에서 개념 자체를 다루지 않았다는 뜻은 아닙니다.

- 페이지별 청킹을 싼타페 PDF 전체에 적용한 초기 길이 실험입니다.
- 700자·100자 겹침은 당시 비교 기준이며 현재 .py의 최종 기준은 실제 모델 입력 128토큰입니다.

전체 목록: [수업 기반·추가 적용 기록](../src/car_search_rag/zzong_santafe_lag/learning_additions.md)


In [2]:
# [프로젝트 추가 기록] 페이지별 청킹을 싼타페 PDF 전체에 적용한 초기 길이 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
from pathlib import Path
from pypdf import PdfReader

# PDF 위치를 정합니다.
project_folder = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
pdf_path = project_folder / 'data' / 'santafe_hev_manual.pdf'
reader = PdfReader(str(pdf_path))
print(f'PDF 경로: {pdf_path}')
print(f'전체 페이지 수: {len(reader.pages)}')

PDF 경로: C:\mle-02-p1-team1\data\santafe_hev_manual.pdf
전체 페이지 수: 779


## 청크 나누기 기준

한 페이지에서 최대 700자 정도씩 나누고, 앞뒤 문맥이 끊기지 않도록 100자 정도 겹칩니다. 문장이나 줄이 끝나는 위치를 우선 찾습니다. 그림이 있는 페이지는 그림 정보를 첫 청크에 붙입니다. 페이지가 여러 청크로 나뉘면 그림과 글의 정확한 연결은 나중에 확인해야 합니다.

In [4]:
# [프로젝트 추가 기록] 페이지별 청킹을 싼타페 PDF 전체에 적용한 초기 길이 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
from pathlib import Path
from statistics import mean, median
from pypdf import PdfReader

# PDF 경로를 정하고 파일을 엽니다.
project_folder = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
pdf_path = project_folder / "data" / "santafe_hev_manual.pdf"
reader = PdfReader(str(pdf_path))
print(f"PDF 경로: {pdf_path}")
print(f"전체 페이지 수: {len(reader.pages)}")

# 사용자가 앞서 확인한 그림 설명만 넣습니다.
known_image_descriptions = {
    30: {"I1.jpg": "차량 외부 I의 주요 부품 위치를 번호로 표시한 안내도."},
    31: {"I1.jpg": "차량 외부 II의 주요 부품 위치를 번호로 표시한 안내도."},
    33: {"I1.jpg": "차량 내부 주요 부품의 위치를 번호로 표시한 안내도. A와 B 영역은 조작부를 확대해 보여준다."},
    34: {"I1.jpg": "차량 내부 II의 주요 부품 위치를 번호로 표시한 안내도."},
    35: {"I1.jpg": "차량 내부 III의 주요 부품 위치를 번호로 표시한 안내도."},
    44: {"I1.jpg": "온도 눈금이 -30도부터 50도까지 표시되어 있다.", "I2.jpg": "1.6 T-GDi HEV 행에 SAE 점도 0W-20이 표시되어 있다."},
    45: {"I1.jpg": "화살표가 엔진과 차량 실내 사이 차체 패널의 차대번호 타각 위치를 가리킨다."},
    52: {"I2.jpg": "뜨거운 용기 위로 손을 대지 말라는 X 표시가 있다."},
    74: {"I2.jpg": "번호 1~4가 표시된 좌석 조절 스위치와 좌석 움직임 방향을 보여준다.", "I7.jpg": "좌석의 앞뒤 위치, 높이, 등받이 기울기 조절 방향을 보여준다."},
}

def split_page_text(text, max_chars=700, overlap_chars=100):
    # 글이 길면 문장이나 줄이 끝나는 위치를 찾아 나눕니다.
    text = text.strip()
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            lower_bound = start + int(max_chars * 0.55)
            boundaries = [text.rfind("\n", lower_bound, end), text.rfind(". ", lower_bound, end), text.rfind("다. ", lower_bound, end)]
            boundary = max(boundaries)
            if boundary >= lower_bound:
                end = boundary + 1
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= len(text):
            break
        # 앞 청크의 문맥 100자를 다음 청크에 겹쳐 둡니다.
        start = max(start + 1, end - overlap_chars)
        while start < len(text) and text[start].isspace():
            start += 1
    return chunks

chunk_records, image_manifest = [], []
text_page_count = image_page_count = 0
multi_chunk_image_pages = []
known_manual_pages = {30:25, 31:26, 33:28, 34:29, 35:30, 44:39, 45:40, 51:46, 52:47, 53:48, 72:67, 74:69}

for pdf_page_number, page in enumerate(reader.pages, start=1):
    raw_text = (page.extract_text() or "").strip()
    page_images = []
    try:
        page_image_objects = page.images
    except Exception as error:
        page_image_objects = []
        image_manifest.append({"pdf_page_number": pdf_page_number, "error": str(error)})

    # 한 이미지가 오류를 내도 나머지 이미지를 계속 읽습니다.
    for image in page_image_objects:
        try:
            image_name = image.name
            image_path = project_folder / "data" / "zzong_santafe_lag" / "images" / f"pdf_page_{pdf_page_number}" / image_name
            relative_path = image_path.relative_to(project_folder).as_posix() if image_path.exists() else None
            image_part = {"name": image_name, "size": image.image.size, "local_path": relative_path, "description": known_image_descriptions.get(pdf_page_number, {}).get(image_name)}
            page_images.append(image_part)
            image_manifest.append({"pdf_page_number": pdf_page_number, **image_part})
        except Exception as error:
            image_manifest.append({"pdf_page_number": pdf_page_number, "error": str(error)})

    text_page_count += bool(raw_text)
    image_page_count += bool(page_images)
    page_chunks = split_page_text(raw_text)
    if not page_chunks and page_images:
        # 글자가 없는 그림 페이지도 출처를 가진 청크로 남깁니다.
        page_chunks = [""]
    if page_images and len(page_chunks) > 1:
        multi_chunk_image_pages.append((pdf_page_number, len(page_chunks), len(page_images)))

    for chunk_index, raw_chunk in enumerate(page_chunks):
        descriptions = [im["description"] for im in page_images if im.get("description")] if chunk_index == 0 else []
        content = raw_chunk
        if descriptions:
            content = (content + "\n[그림 설명] " + " ".join(descriptions)).strip()
        if not content and page_images:
            content = "그림이 포함된 페이지이며, 그림 내용을 추가 확인해야 합니다."
        content_type = "figure_with_text" if page_images and raw_text else ("image_only" if page_images else "instruction_text")
        chunk_records.append({"content": content, "raw_text": raw_chunk, "metadata": {"pdf_page_number": pdf_page_number, "manual_page_number": known_manual_pages.get(pdf_page_number), "chunk_index": chunk_index, "content_type": content_type, "image_linkage_status": "page_level_needs_review" if page_images else None}, "image_parts": page_images if chunk_index == 0 else []})

lengths = [len(item["content"]) for item in chunk_records]
captioned_images = sum(bool(im.get("description")) for im in image_manifest if "name" in im)
image_elements = sum("name" in im for im in image_manifest)
print(f"글자가 있는 페이지: {text_page_count}")
print(f"청크 기록 수: {len(chunk_records)}")
print(f"청크 글자 수 평균 / 중앙값 / 최댓값: {mean(lengths):.1f} / {median(lengths):.1f} / {max(lengths)}")
print(f"이미지가 있는 페이지: {image_page_count}")
print(f"확인된 이미지 요소 수: {image_elements}")
print(f"설명이 등록된 이미지 수: {captioned_images}")
print(f"설명이 아직 없는 이미지 수: {image_elements - captioned_images}")
print(f"이미지 연결 위치를 추가 검토할 페이지 수: {len(multi_chunk_image_pages)}")
print("저장 위치: 메모리만 사용 (청크 파일/Supabase에 저장하지 않음)")

PDF 경로: C:\mle-02-p1-team1\data\santafe_hev_manual.pdf
전체 페이지 수: 779
글자가 있는 페이지: 777
청크 기록 수: 1279
청크 글자 수 평균 / 중앙값 / 최댓값: 498.1 / 560.0 / 732
이미지가 있는 페이지: 509
확인된 이미지 요소 수: 864
설명이 등록된 이미지 수: 11
설명이 아직 없는 이미지 수: 853
이미지 연결 위치를 추가 검토할 페이지 수: 121
저장 위치: 메모리만 사용 (청크 파일/Supabase에 저장하지 않음)


## 일부 청크 확인

PDF 44, 45, 52, 74쪽의 글, 출처, 이미지 정보가 함께 유지되는지 확인합니다.

In [6]:
# [프로젝트 추가 기록] 페이지별 청킹을 싼타페 PDF 전체에 적용한 초기 길이 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
for page_number in [44, 45, 52, 74]:
    page_items = [item for item in chunk_records if item["metadata"]["pdf_page_number"] == page_number]
    print(f"\n--- PDF {page_number}쪽: {len(page_items)}개 청크 ---")
    for item in page_items[:2]:
        print("출처:", item["metadata"])
        print("원문 앞부분:", item["raw_text"][:220].replace("\n", " "))
        print("검색용 글 끝부분:", item["content"][-220:].replace("\n", " "))
        print("이미지 정보:", item["image_parts"])
uncaptioned_pages = sorted({im["pdf_page_number"] for im in image_manifest if "name" in im and not im.get("description")})
print(f"설명이 필요한 PDF 페이지 수: {len(uncaptioned_pages)}")
print("설명 없는 이미지가 있는 페이지 앞부분:", uncaptioned_pages[:30])


--- PDF 44쪽: 1개 청크 ---
출처: {'pdf_page_number': 44, 'manual_page_number': 39, 'chunk_index': 0, 'content_type': 'figure_with_text', 'image_linkage_status': 'page_level_needs_review'}
원문 앞부분: 온도에 따른 엔진 오일 SAE 점도 분류표 SAE 점도와 사용 온도 범위 온도 (℃) 1.6 T-GDi HEV 안내 및 차량 정보 39
검색용 글 끝부분: 온도에 따른 엔진 오일 SAE 점도 분류표 SAE 점도와 사용 온도 범위 온도 (℃) 1.6 T-GDi HEV 안내 및 차량 정보 39 [그림 설명] 온도 눈금이 -30도부터 50도까지 표시되어 있다. 1.6 T-GDi HEV 행에 SAE 점도 0W-20이 표시되어 있다.
이미지 정보: [{'name': 'I1.jpg', 'size': (945, 71), 'local_path': 'data/zzong_santafe_lag/images/pdf_page_44/I1.jpg', 'description': '온도 눈금이 -30도부터 50도까지 표시되어 있다.'}, {'name': 'I2.jpg', 'size': (945, 71), 'local_path': 'data/zzong_santafe_lag/images/pdf_page_44/I2.jpg', 'description': '1.6 T-GDi HEV 행에 SAE 점도 0W-20이 표시되어 있다.'}]

--- PDF 45쪽: 1개 청크 ---
출처: {'pdf_page_number': 45, 'manual_page_number': 40, 'chunk_index': 0, 'content_type': 'figure_with_text', 'image_linkage_status': 'page_level_needs_review'}
원문 앞부분: 차대번호 (VIN) 2C_VINPosition 차대번호(VIN: Vehicle Identifica